# Compare preprocessing candidates

Every candidate on the same 10 dark images (one per class), so differences come only from preprocessing.

| step | lecture notes (PDF page) | what it does |
|---|---|---|
| contrast stretch | §3.3.4 (p.50) | linear stretch of the 1st–99th percentile of L to 0–255 |
| histogram equalization | §3.4 (p.55–58) | global equalization of L |
| CLAHE | §3.4.2 (p.58–60) | local equalization of L, clip 2.0, 8×8 tiles |
| gamma | §3.3.3 (p.49) | `s = r^γ`, γ per image so mean L → 0.5 (only brightens, γ ≥ 0.4) |
| unsharp mask | named only (p.97) | `L + 1·(L − Gaussian blur)` |
| high-boost | exercise 8 (p.100), solution p.393 | `L + k·(L − Gaussian blur)`, k = 2 |
| wavelet | idea only, one line (p.146); BayesShrink: Chang et al., IEEE TIP 2000 | BayesShrink soft threshold (all channels, in YCbCr) |

All steps except wavelet change only L of LAB, so food colours (a, b) stay the same.
`A → B` means A first, then B.

Images come from **official train** only, resized to short side 256 like `01_original`. Looking only: don't pick parameters by looking at test images, and don't choose the final dataset from these 10 images. That decision needs val results.

In [ ]:
SEED = 0           # change to draw 10 other images
DARK_ONLY = True   # draw from images with mean grey < 80

CLAHE_CLIP, CLAHE_TILE = 2.0, 8          # same as pipeline_02.md
STRETCH_PCT = (1, 99)
GAMMA_MIN = 0.4
SHARPEN_SIGMA = 2.0                      # Gaussian blur used by unsharp / high-boost
HIGH_BOOST_K = 2.0
WAVELET = dict(method="BayesShrink", mode="soft", rescale_sigma=True)

In [ ]:
import os
import random
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from skimage.restoration import denoise_wavelet, estimate_sigma

# ---- paths (preset) -------------------------------------------------------
# project root = the folder that contains pipeline_preprocess/ (works from any sub-folder)
PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pipeline_preprocess").is_dir())
DATA_DIR = Path(os.environ.get("FOOD_DATA_DIR", PROJECT_ROOT / "data"))
IMG_DIR = DATA_DIR / "raw_data" / "images"
TRAIN_TXT = next((DATA_DIR / "raw_data").glob("**/train.txt"))

CLASSES = ["bibimbap", "miso_soup", "mussels", "prime_rib", "edamame",
           "baby_back_ribs", "paella", "chicken_wings", "guacamole", "nachos"]

In [ ]:
# one image per class, official train only
train_keys = set(TRAIN_TXT.read_text().split())
bright = pd.read_csv(DATA_DIR / "dataset" / "meta" / "brightness.csv", dtype={"id": str})
bright = bright[(bright.label + "/" + bright.id).isin(train_keys)]
if DARK_ONLY:
    bright = bright[bright.mean_grey < 80]

rng = random.Random(SEED)
picked = [(c, rng.choice(sorted(bright[bright.label == c].id))) for c in CLASSES]
picked

## Steps

In [ ]:
def resize_short(img, short=256):
    h, w = img.shape[:2]
    s = short / min(h, w)
    return img if s >= 1 else cv2.resize(img, (round(w * s), round(h * s)), interpolation=cv2.INTER_AREA)


def on_lightness(f):
    """Wrap f(L uint8) -> L uint8 into an RGB step that keeps a, b of LAB."""
    def step(rgb):
        l, a, b = cv2.split(cv2.cvtColor(rgb, cv2.COLOR_RGB2LAB))
        return cv2.cvtColor(cv2.merge([f(l), a, b]), cv2.COLOR_LAB2RGB)
    return step


def to_u8(x):
    return np.clip(x, 0, 255).round().astype(np.uint8)


@on_lightness
def stretch(l):
    lo, hi = np.percentile(l, STRETCH_PCT)
    return to_u8((l.astype(np.float32) - lo) * 255 / max(hi - lo, 1))


@on_lightness
def histeq(l):
    return cv2.equalizeHist(l)


@on_lightness
def clahe(l):
    return cv2.createCLAHE(clipLimit=CLAHE_CLIP, tileGridSize=(CLAHE_TILE, CLAHE_TILE)).apply(l)


@on_lightness
def gamma(l):
    r = l.astype(np.float32) / 255
    g = np.clip(np.log(0.5) / np.log(np.clip(r.mean(), 1e-3, 0.999)), GAMMA_MIN, 1.0)
    return to_u8(255 * r ** g)


def sharpen(k):
    @on_lightness
    def step(l):
        l = l.astype(np.float32)
        return to_u8(l + k * (l - cv2.GaussianBlur(l, (0, 0), SHARPEN_SIGMA)))
    return step


def wavelet(rgb):
    den = denoise_wavelet(rgb / 255.0, channel_axis=-1, convert2ycbcr=True, **WAVELET)
    return to_u8(den * 255)


unsharp, high_boost = sharpen(1.0), sharpen(HIGH_BOOST_K)
then = lambda *fs: lambda x: [x := f(x) for f in fs][-1]

STAGES = {
    "original": lambda x: x,
    # single steps
    "contrast stretch": stretch,
    "hist-eq": histeq,
    "CLAHE": clahe,
    "gamma": gamma,
    "unsharp": unsharp,
    "high-boost": high_boost,
    # combinations
    "stretch → CLAHE": then(stretch, clahe),
    "unsharp → CLAHE": then(unsharp, clahe),
    "CLAHE → unsharp": then(clahe, unsharp),
    "high-boost → CLAHE": then(high_boost, clahe),
    "wavelet → CLAHE": then(wavelet, clahe),
}
images = []
for cls, img_id in picked:
    rgb = resize_short(cv2.cvtColor(cv2.imread(str(IMG_DIR / cls / f"{img_id}.jpg")), cv2.COLOR_BGR2RGB))
    images.append({name: f(rgb) for name, f in STAGES.items()})

## Numbers per step (mean over the 10 images)

- **brightness**: mean L (0–255). Higher = brighter.
- **contrast**: standard deviation of L. Higher = more global contrast.
- **sharpness**: variance of the Laplacian of L. Higher = stronger edges and texture (and noise).
- **noise σ**: `estimate_sigma` from the finest wavelet band. On JPEG photos it also counts fine texture, so read it together with sharpness.

A good step for dark images raises brightness and contrast while keeping noise σ low.

In [ ]:
def measures(rgb):
    l = cv2.cvtColor(rgb, cv2.COLOR_RGB2LAB)[..., 0]
    return {"brightness": l.mean(), "contrast": l.std(),
            "sharpness": cv2.Laplacian(l, cv2.CV_64F).var(),
            "noise σ": estimate_sigma(rgb, channel_axis=-1, average_sigmas=True)}


table = pd.DataFrame({name: pd.DataFrame([measures(s[name]) for s in images]).mean() for name in STAGES}).T
table.round(2)

## Whole images
One figure per image, all steps side by side.

In [ ]:
cols = 4
for (cls, img_id), stages in zip(picked, images):
    fig, axes = plt.subplots(len(STAGES) // cols, cols, figsize=(13, 3.3 * len(STAGES) // cols))
    for ax, (name, img) in zip(axes.ravel(), stages.items()):
        ax.imshow(img)
        ax.set_title(name, fontsize=11)
        ax.axis("off")
    fig.suptitle(f"{cls} {img_id}", fontsize=15, fontweight="bold")
    fig.tight_layout()
    plt.show()

## Zoom: darkest patch and busiest patch

Two 64×64 patches per image:
- **dark**: the darkest patch of the original. Shows how much each step amplifies noise in flat dark areas.
- **detail**: the patch with the most edges. Shows how much texture each step keeps or adds.

In [ ]:
def patch_at(rgb, size, score):
    grey = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY).astype(np.float32)
    m = cv2.blur(score(grey), (size, size))[size // 2:-size // 2, size // 2:-size // 2]
    return np.unravel_index(m.argmin(), m.shape)


PATCHES = {"dark": lambda g: g, "detail": lambda g: -np.abs(cv2.Laplacian(g, cv2.CV_32F))}
for kind, score in PATCHES.items():
    fig, axes = plt.subplots(len(images), len(STAGES), figsize=(1.6 * len(STAGES), 1.75 * len(images)))
    for r, ((cls, img_id), stages) in enumerate(zip(picked, images)):
        y, x = patch_at(stages["original"], 64, score)
        for c, (name, img) in enumerate(stages.items()):
            ax = axes[r, c]
            ax.imshow(img[y:y + 64, x:x + 64], interpolation="nearest")
            ax.set_xticks([]); ax.set_yticks([])
            if r == 0:
                ax.set_title(name, fontsize=8)
            if c == 0:
                ax.set_ylabel(cls, fontsize=8)
    fig.suptitle(f"{kind} patch (64×64)", fontsize=14, fontweight="bold")
    fig.tight_layout()
    plt.show()